In [11]:
import pandas as pd
from pathlib import Path
import sys
sys.path.append("../src")
from preprocessing import create_preprocessor

FE_DF_PATH = Path("../data/processed/feature_engineered_df.parquet")
fe_df = pd.read_parquet(FE_DF_PATH)

In [12]:
# look at the dataframe
fe_df.head()

,Store,DayOfWeek,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,...,Month,Day,WeekOfYear,Quarter,IsWeekend,Season,CompetitionExists,WasPromo2Active,CompetitionAgeMonths,Sales
0,1,5,555,1,1,0,1,c,a,1270,...,7,31,31,3,0,Summer,1,0,82,5263
1,2,5,625,1,1,0,1,a,a,570,...,7,31,31,3,0,Summer,1,1,92,6064
2,3,5,821,1,1,0,1,a,a,14130,...,7,31,31,3,0,Summer,1,1,103,8314
3,4,5,1498,1,1,0,1,c,c,620,...,7,31,31,3,0,Summer,1,0,70,13995
4,5,5,559,1,1,0,1,a,a,29910,...,7,31,31,3,0,Summer,1,0,3,4822


In [13]:
# shape
fe_df.shape

(1017209, 27)

In [14]:
# check NaN values
assert fe_df.isna().sum().sum() == 0

In [15]:
# select features
model_df = fe_df[["Store", "Customers", "Day", "WeekOfYear", "Quarter", "Open", "Season", 
           "Promo", "SchoolHoliday", "CompetitionDistance", "CompetitionAgeMonths", 
           "WasPromo2Active", "StoreType", "Year", "Month", 
           "DayOfWeek", "StateHoliday", "Assortment", "PromoInterval", "Sales"]]

In [16]:
# create train and test splits
# train: all of 2013, 2014 and Jan. to May 2015
# test: June-July 2015
train = model_df[(model_df.Year < 2015) | ((model_df.Year == 2015) & (model_df.Month < 6))]
valid = model_df[(model_df.Year == 2015) & (model_df.Month >= 6)]

X_train = train.drop(columns="Sales")
y_train = train.Sales

X_valid = valid.drop(columns="Sales")
y_valid = valid.Sales

In [17]:
# select numerical and categorical features
num_features = X_train.select_dtypes(include="number").columns
cat_features = X_train.select_dtypes(exclude="number").columns

In [18]:
# create pipelines
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

preprocessor = create_preprocessor(num_features, cat_features)

pipelines = {
    "Dummy": Pipeline([
        ("preprocessor", preprocessor),
        ("regressor", DummyRegressor(strategy="median")),
    ]),
    "Linear Regression": Pipeline([
        ("preprocessor", preprocessor),
        ("regressor", LinearRegression(n_jobs=-1)),
    ]),
    "Random Forest": Pipeline([
        ("preprocessor", preprocessor),
        ("regressor", RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)),
    ]),
    "XG Boost": Pipeline([
        ("preprocessor", preprocessor),
        ("regressor", XGBRegressor(n_estimators=200, random_state=42, n_jobs=-1, objective="reg:squarederror", eval_metric="rmse")),
    ]),
}

In [19]:
# metrics
import joblib
from sklearn.model_selection import cross_val_score, TimeSeriesSplit
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

results = {}
tscv = TimeSeriesSplit(n_splits=5)
for model_name, pipeline in pipelines.items():
    cv_scores = cross_val_score(pipeline,
                                X_train,
                                y_train,
                                cv=tscv,
                                scoring="r2",
                                )
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_valid)
    
    results[model_name] = {
        "CV Mean R²": cv_scores.mean(),
        "CV Std": cv_scores.std(),
        "Test R² score": r2_score(y_valid, y_pred),
        "MAE": mean_absolute_error(y_valid, y_pred),
        "MSE": mean_squared_error(y_valid, y_pred),
        "MAPE": mean_absolute_percentage_error(y_valid, y_pred)
    }
    filename = model_name.lower().replace(" ", "_") + ".joblib"
    joblib.dump(pipeline, f"../outputs/models/{filename}")

In [20]:
# save the metrics df
metrics_df = pd.DataFrame(results).T.round(2).reset_index(names="Model")
metrics_df.to_csv("../data/processed/metrics_df.csv", index=False)